# 07 - Conversation

Manual/exploratory multi-turn test against the real `src/migrantbuddy` conversational
pipeline (`rag/graph.py`) -- unlike `01`-`06`, this doesn't prototype-before-extraction
(the LangGraph conversation layer was built directly in `src/`, no notebook came
first). This notebook exists to *watch* the graph's internal state evolve turn by
turn -- the rewritten standalone query, the retrieved context, and the running
summary once it kicks in -- which `tests/test_rag_graph.py` checks with assertions
but doesn't let you eyeball.

Requires the real data pipeline output to already exist (`data/processed/chunks.json`
+ the Chroma collection) and Ollama running locally. Hits the real generation
backend multiple times per turn (query rewrite + generate, plus summarize once
triggered) -- expect this to take a few minutes to run end to end, not seconds.

## Step 1: Setup

Load the real chunks + Chroma collection, build a real `Retriever`, and compile the
graph directly (not via `ConversationService`) so we can call `graph.get_state(config)`
after each turn -- `ConversationService.answer()` only returns the final answer +
sources, not the intermediate `standalone_query`/`summary`/`context_chunks` fields
this notebook wants to inspect.

In [1]:
import json

import chromadb
from migrantbuddy.config import CHROMA_COLLECTION_NAME, CHROMA_DIR, PROCESSED_DIR
from migrantbuddy.indexing import Chunk
from migrantbuddy.rag.graph import build_graph
from migrantbuddy.retrieval import Retriever

chunks_path = PROCESSED_DIR / "chunks.json"
raw_chunks = json.loads(chunks_path.read_text(encoding="utf-8"))
chunks = [Chunk(**item) for item in raw_chunks]

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name=CHROMA_COLLECTION_NAME)
retriever = Retriever(chunks, collection)

graph = build_graph(retriever)

len(chunks), collection.count()

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4597.90it/s]


(16, 16)

## Step 2: The multi-turn scenario

7 turns, one `thread_id`, designed to exercise the parts a single-turn test can't:

- **Turn 2** is a bare follow-up ("What about for daily-rated workers?") with almost
  no signal on its own -- tests query rewriting.
- **Turns 3, 5, 6** change topic entirely (salary timing, medical insurance, work
  permit conditions) -- keeps the conversation realistic, not just one topic looped.
- **Turn 7** asks the model to recall turn 1's specific figure ($3000 salary, the
  overtime rate) after enough turns have passed that `SUMMARY_TRIGGER_MESSAGE_COUNT`
  (12 messages / 6 exchanges) should have been crossed -- tests whether the summary
  actually preserves that detail instead of losing it once it's no longer verbatim.

In [2]:
THREAD_ID = "notebook-multiturn-demo"

TURNS = [
    ("How much overtime pay am I entitled to if my salary is $3000?", "establishes context: salary=$3000"),
    ("What about for daily-rated workers?", "bare follow-up -- tests query rewriting"),
    ("When must my employer pay my salary?", "topic change"),
    ("What if I resign without notice?", "follow-up on the new topic"),
    ("How much medical insurance must my employer provide?", "topic change"),
    ("What happens to my work permit when my contract ends?", "topic change -- should push past the summary threshold"),
    ("Going back to my first question, what was the overtime rate again?", "tests whether the summary preserved turn 1's detail"),
]

[t[0] for t in TURNS]

['How much overtime pay am I entitled to if my salary is $3000?',
 'What about for daily-rated workers?',
 'When must my employer pay my salary?',
 'What if I resign without notice?',
 'How much medical insurance must my employer provide?',
 'What happens to my work permit when my contract ends?',
 'Going back to my first question, what was the overtime rate again?']

## Step 3: Run the conversation, inspecting state after every turn

After each `graph.invoke(...)`, pull the full state back via `graph.get_state(config)`
-- this is what actually exposes `standalone_query`, `summary`, and `context_chunks`,
none of which `ConversationService.answer()`'s return value carries.

In [3]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": THREAD_ID}}

for i, (message, note) in enumerate(TURNS, start=1):
    graph.invoke({"messages": [HumanMessage(content=message)]}, config=config)
    snapshot = graph.get_state(config)
    state = snapshot.values

    answer = state["messages"][-1].content
    sources = state.get("sources", [])
    standalone_query = state.get("standalone_query", "")
    summary = state.get("summary", "")

    print(f"\n{'=' * 80}\nTurn {i}: {message}\n({note})\n{'=' * 80}")
    print(f"\nRewritten standalone query: {standalone_query!r}")
    print(f"Retrieved sources: {sources}")
    print(f"\nAnswer (first 300 chars):\n{answer[:300]}")
    print(f"\nMessages currently in state: {len(state['messages'])}")
    print(f"Current summary: {summary[:300] if summary else '(none yet)'}")

Deserializing unregistered type migrantbuddy.indexing.chunking.Chunk from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('migrantbuddy.indexing.chunking', 'Chunk')]



Turn 1: How much overtime pay am I entitled to if my salary is $3000?
(establishes context: salary=$3000)

Rewritten standalone query: 'How much overtime pay am I entitled to if my salary is $3000?'
Retrieved sources: ['https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/salary/paying-salary', 'https://www.mom.gov.sg/employment-practices/salary/paying-salary']

Answer (first 300 chars):
Since your monthly basic salary ($3,000) exceeds both limits mentioned:
- Non-workmen limit of $2,600
- Workmen limit of $4,500 

...you do not qualify for overtime pay under these rules. However, you might have other rights or contracts that could apply - check with TADM (Tripartite Alliance for Di

Messages currently in state: 2
Current summary: (none yet)

Tur

## Step 4: Final state -- confirm summarization actually happened

By turn 7, `SUMMARY_TRIGGER_MESSAGE_COUNT` (12) should have been crossed at least
once: `summary` should be non-empty, and `messages` should hold fewer than the full
14 messages 7 turns would produce unbounded (7 human + 7 AI) -- confirming older
turns actually got folded into the summary instead of just accumulating forever.

In [4]:
final_state = graph.get_state(config).values

print(f"Final message count: {len(final_state['messages'])} (unbounded would be 14)")
print(f"\nFull summary:\n{final_state.get('summary', '(none)')}")
print(f"\nMessages kept verbatim:")
for m in final_state["messages"]:
    role = "assistant" if m.type == "ai" else "user"
    print(f"  [{role}] {m.content[:100]}")

Final message count: 7 (unbounded would be 14)

Full summary:
**Summary of Singapore Employment Rules Conversation**

1. **Overtime Pay**: 
   - Non-workmen: Eligible only if monthly salary ≤ $2,600
   - Workmen: Eligible only if monthly salary ≤ $4,500
   - Your salary ($3,000) is above the threshold → No overtime pay entitlement

2. **Daily-Rated Workers**:
   - Overtime rate = Daily basic wage / Regular working hours
   - Example: $50 daily wage × 1.5 for extra hours
   - Pay within 14 days of period end
   - Maximum daily work: 12 hours (exceptions require employer approval)

3. **Salary Payment Rules**:
   - Minimum once monthly
   - Deadline: Within 7 days after pay period ends (regular salary)
   - Overtime deadline: Within 14 days after pay period ends
   - Payment must be during working hours via bank transfer or check

4. **Resignation Without Notice**:
   - Last day of work = Resignation date (no notice required in this scenario)

Messages kept verbatim:
  [assistant] If you